## 2.4 记忆治理策略（上下文管理）
随着对话的进行，历史消息不断累积， `state会持续增长` ，为模型带来挑战：
1. LLM的 上下文窗口是有限的 ，完整历史可能无法装入LLM的上下文窗口，导致上下文丢失或错
误。
2. 即便模型的上下文窗口够大，多数LLM在长上下文场景仍然表现不佳。模型会 `被陈旧或离题的内容`“分散注意力” 。
3. 同时，会带来 高昂的token花费 。

此时需要对上下文进行管理：对历史记录进行压缩、清理、重组等。

### 2.4.1 消息裁剪
调用模型前裁剪上下文。

目标是控制token用量，通常 保留系统初始消息和最近若干消息 ，或 按token数保留末尾内容 。

适合成本敏感、对旧上下文依赖不强的场景。

In [9]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.tools import tool
from pydantic import BaseModel, Field

load_dotenv(override=True, verbose=True, dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

custom_profile = {
    "max_input_tokens": 128_000
}

model = ChatOpenAI(
    model="gpt-5.6-sol",
    api_key=os.environ["OPENAI_API_KEY"],
    base_url=os.environ["OPENAI_API_BASE"],
    #profile=custom_profile
)

In [3]:
from langchain_core.messages import HumanMessage
from langchain.messages import RemoveMessage
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import before_model
from langgraph.runtime import Runtime
from langchain_core.runnables import RunnableConfig
from typing import Any

@before_model
def trim_messages(state:AgentState,runtime:Runtime) -> dict[str, Any] | None:

    messages = state["messages"]

    if len(messages) <= 3:
        return None

    first_message = messages[0]
    # 如果有偶数条消息，则取最近的3条消息；如果有奇数条消息，则取最近的4条消息
    recent_messages = messages[-3:] if len(messages) % 2 == 0 else messages[-4:]
    # 将第一条消息和最近的消息组合成新的消息列表
    new_messages = [first_message] + recent_messages
    # 返回一个字典，包含一个键 "messages"，其值是一个列表，列表中包含一个 RemoveMessage 对象和新的消息列表
    return {
        "messages": [
            RemoveMessage(id=REMOVE_ALL_MESSAGES),
            *new_messages
        ],
    }

agent = create_agent(
    model=model,
    middleware=[trim_messages],
    checkpointer=InMemorySaver(),
)

config: RunnableConfig = {"configurable": {"thread_id": "1"}}

agent.invoke({"messages": [HumanMessage("你好，我是老王")]}, config)
agent.invoke({"messages": [HumanMessage("从现在起，你叫小王")]}, config)
agent.invoke({"messages": [HumanMessage("今天天气不错")]}, config)
agent.invoke({"messages": [HumanMessage("但是深圳下了很大的雨")]}, config)
agent.invoke({"messages": [HumanMessage("北京下了很大的雪")]}, config)
final_response = agent.invoke({"messages": [HumanMessage("告诉我，你是谁？我是谁？")]}, config)

for msg in final_response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

你好，我是老王
================================== Ai Message ==================================

那确实不太像“天气不错”了。深圳这会儿雨大，出门记得带伞，留意积水和路况。
================================ Human Message =================================

北京下了很大的雪
================================== Ai Message ==================================

北京也在下大雪啊，那出门更要注意路滑和能见度了。深圳是暴雨，北京是大雪，今天两边都不太适合随便出门。
================================ Human Message =================================

告诉我，你是谁？我是谁？
================================== Ai Message ==================================

我是 Codex，一个协助你处理问题和代码的 AI 助手。你刚才告诉我：“你好，我是老王”，所以你是老王。


langchain API 实际提供了裁剪消息的函数:

```python
trim_messages(
  messages: Iterable[MessageLikeRepresentation] | PromptValue,
  *,
  max_tokens: int,
  token_counter: Callable[[list[BaseMessage]], int] | Callable[[BaseMessage], int] | BaseLanguageModel[Any] | Literal['approximate'],
  strategy: Literal['first', 'last'] = 'last',
  allow_partial: bool = False,
  end_on: str | type[BaseMessage] | Sequence[str | type[BaseMessage]] | None = None,
  start_on: str | type[BaseMessage] | Sequence[str | type[BaseMessage]] | None = None,
  include_system: bool = False,
  text_splitter: Callable[[str], list[str]] | TextSplitter | None = None
) -> list[BaseMessage]
```


In [6]:
from langchain_core.messages import (
    AIMessage,
    HumanMessage,
    BaseMessage,
    SystemMessage,
    trim_messages,
)

messages = [
    SystemMessage("you're a good assistant, you always respond with a joke."),
    HumanMessage("i wonder why it's called langchain"),
    AIMessage(
        'Well, I guess they thought "WordRope" and "SentenceString" just '
        "didn't have the same ring to it!"
    ),
    HumanMessage("and who is harrison chasing anyways"),
    AIMessage(
        "Hmmm let me think.\n\nWhy, he's probably chasing after the last "
        "cup of coffee in the office!"
    ),
    HumanMessage("what do you call a speechless parrot"),
]

trim_messages(
    messages,
    max_tokens=45,
    strategy="last",
    token_counter=ChatOpenAI(model="gpt-5.5"),
    # Most chat models expect that chat history starts with either:
    # (1) a HumanMessage or
    # (2) a SystemMessage followed by a HumanMessage
    start_on="human",
    # Usually, we want to keep the SystemMessage
    # if it's present in the original history.
    # The SystemMessage has special instructions for the model.
    include_system=True,
    allow_partial=False,
)

[SystemMessage(content="you're a good assistant, you always respond with a joke.", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='what do you call a speechless parrot', additional_kwargs={}, response_metadata={})]

#### 补充 1：trim_messages 的参数与返回值

`trim_messages` 只做一件事：**输入一个消息列表，按 token 数（或条数）从中挑出一部分，返回一个新列表**。它不修改原列表，也不涉及 state 和 checkpointer，就是一个普通函数。

| 参数 | 含义 | 常用取值 |
|---|---|---|
| `messages` | 要裁剪的消息列表 | `state["messages"]` 或 `request.messages` |
| `max_tokens` | 裁剪结果最多允许多少 token | 按模型的上下文窗口和预算设置 |
| `token_counter` | 怎么数 token | `"approximate"`：按字符数估算，速度快。官方推荐在每次调用模型都要运行的地方使用<br>模型对象，如 `ChatOpenAI(...)`：调用它的 `get_num_tokens_from_messages()`，用 tiktoken 在本地计算，**不会请求模型**<br>`len`：每条消息算 1 个，这时 `max_tokens` 就是“最多保留几条消息” |
| `strategy` | 保留哪一头 | `"last"`（默认）：保留最近的消息；`"first"`：保留最早的消息 |
| `include_system` | 开头的 `SystemMessage` 是否一定保留 | `True`：先扣掉系统消息的 token，剩下的预算再分给其他消息。只能和 `"last"` 一起用 |
| `start_on` | 结果必须从哪种消息开始 | `"human"`：把结果开头不是 `HumanMessage` 的消息去掉，保证历史从用户的提问开始。只能和 `"last"` 一起用 |
| `end_on` | 结果必须以哪种消息结束 | 如 `("human", "tool")`。`"last"` 策略下会先去掉末尾不符合的消息，再裁剪 |
| `allow_partial` | 一整条消息放不下时，是否截取它的一部分 | 默认 `False`：整条丢弃 |
| `text_splitter` | `allow_partial=True` 时怎么切分文本 | 默认按换行切分 |

**返回值**：`list[BaseMessage]`，裁剪后的新列表，里面还是原来的那些消息对象，只是少了一部分。另外，如果不传 `messages`、只传其他参数，`trim_messages(...)` 返回的是一个 Runnable，可以像第六章的链那样用 `|` 组合。

`strategy="last"` 的裁剪过程（源码 `langchain_core/messages/utils.py` 中的 `_last_max_tokens`）：

1. 如果 `include_system=True` 且第一条是系统消息，先把它拿出来，`max_tokens` 减去它的 token 数，得到剩余预算；
2. 从最后一条往前，在剩余预算内尽量多地保留消息（用二分查找算出最多能保留几条）；
3. 如果设置了 `start_on`，去掉保留结果开头类型不符合的消息；
4. 把系统消息放回最前面。

下面用和上一个单元格相同的计数方式，算一算上面例子中每一步的 token 数（在本地计算，不请求模型）：

In [12]:
token_counter = ChatOpenAI(model="gpt-5.5")  # 和上一个单元格相同的计数方式

print("每条消息单独计数：")
for msg in messages:
    print(f"  {type(msg).__name__:<14} {token_counter.get_num_tokens_from_messages([msg]):>3}  {msg.text[:36]!r}")

system_tokens = token_counter.get_num_tokens_from_messages(messages[:1])
print(f"\nmax_tokens=45，系统消息占 {system_tokens}，剩余预算 {45 - system_tokens}")
for k in range(1, 4):
    print(f"  保留最后 {k} 条：{token_counter.get_num_tokens_from_messages(messages[-k:])} tokens")

每条消息单独计数：
  SystemMessage   19  "you're a good assistant, you always "
  HumanMessage    14  "i wonder why it's called langchain"
  AIMessage       32  'Well, I guess they thought "WordRope'
  HumanMessage    14  'and who is harrison chasing anyways'
  AIMessage       28  "Hmmm let me think.\n\nWhy, he's probab"
  HumanMessage    16  'what do you call a speechless parrot'

max_tokens=45，系统消息占 19，剩余预算 26
  保留最后 1 条：16 tokens
  保留最后 2 条：41 tokens
  保留最后 3 条：52 tokens


* 剩余预算 26：保留最后 1 条（16）放得下，最后 2 条（41）就超了。所以只保留最后 1 条，再放回系统消息，就是上一个单元格的结果。
* 多条消息一起计数，结果不等于单条之和（最后 2 条是 41，不是 16 + 28 = 44）：`ChatOpenAI` 计数时给每条消息加 3 个格式 token，给整个列表再加 3 个，单独计数时每条都多算了一次这 3 个。所以 `trim_messages` 每次都把候选的消息列表整体拿去计数。

再看看 `start_on` 和 `token_counter=len` 的效果（不请求模型）：

In [13]:
def show(result):
    return [type(m).__name__.removesuffix("Message") for m in result]

# 预算放宽到 70：剩余 51，能保留最后 2 条（AI + Human）
print("不设 start_on：   ", show(trim_messages(messages, max_tokens=70, strategy="last",
                                         token_counter=token_counter, include_system=True)))
print("start_on='human'：", show(trim_messages(messages, max_tokens=70, strategy="last",
                                         token_counter=token_counter, include_system=True,
                                         start_on="human")))

# token_counter=len：每条消息算 1 个，max_tokens=4 就是最多保留 4 条（含系统消息）
print("按条数保留 4 条： ", show(trim_messages(messages, max_tokens=4, strategy="last",
                                         token_counter=len, include_system=True, start_on="human")))

不设 start_on：    ['System', 'AI', 'Human']
start_on='human'： ['System', 'Human']
按条数保留 4 条：  ['System', 'Human', 'AI', 'Human']


* 不设 `start_on` 时，系统消息之后紧跟的是 `AIMessage`，对话看起来像是从模型的回答开始。很多模型要求历史从用户消息开始，所以通常加上 `start_on="human"`，宁可少保留一条。
* `token_counter=len` 适合“只保留最近 N 条”的需求，比如 2.4.1 的例子。

#### 补充 2：LangChain 内部用到 trim_messages 了吗？

* **2.4.1 的中间件没有用。** 它也叫 `trim_messages`（这个例子来自官方文档），但只是按条数切片 `messages[-3:]`，和 `langchain_core` 的 `trim_messages` 函数只是同名。同名还会互相覆盖：2.4.1 的单元格先定义了中间件 `trim_messages`，上一个单元格又导入了同名函数，从那以后这个名字指的就是函数了。
* **LangChain 没有专门的“裁剪中间件”。** 第八章 01 笔记整理的 16 个内置中间件里，与上下文长度有关的是 `SummarizationMiddleware`（摘要）和 `ContextEditingMiddleware`（清理工具结果），没有专门按 token 裁剪消息的。官方文档的做法就是像 2.4.1 那样，自己写一个 `before_model` 中间件。
* **`SummarizationMiddleware` 内部用到了它。** 生成摘要时，要把需要摘要的旧消息交给摘要模型。如果这些旧消息本身很长，也会超出摘要模型的上下文，所以它先用 `trim_messages` 裁剪一遍（源码 `langchain/agents/middleware/summarization.py`）：

```python
def _trim_messages_for_summary(self, messages):
    if self.trim_tokens_to_summarize is None:     # 构造参数 trim_tokens_to_summarize，默认 4000
        return messages
    return trim_messages(
        messages,
        max_tokens=self.trim_tokens_to_summarize,
        token_counter=self.token_counter,
        start_on="human",
        strategy="last",
        allow_partial=True,
        include_system=True,
    )
```

也就是说，交给摘要模型的最多是最近 4000 个 token 的旧消息。参数组合正是官方推荐的那一套：`strategy="last"` + `start_on="human"` + `include_system=True`。

### 2.4.2 消息删除
消息裁剪强调“在 `模型调用前裁剪` 消息列表，**控制模型可以看到的上下文范围”**，而消息删除强调 模型调用完成后将某些消息从消息列表中移除 ，永久更改状态。

适合明确要遗忘、清理、重置某些历史。

In [7]:
from langchain.messages import RemoveMessage
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import after_model
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.runtime import Runtime
from langchain_core.runnables import RunnableConfig


@after_model
def delete_old_messages(state: AgentState, runtime: Runtime) -> dict | None:
    messages = state["messages"]
    # 保持最近的 5 条消息
    if len(messages) > 5:
        # 框架中通常使用 RemoveMessage 来标记删除，并返回更新状态。
        to_delete = len(messages) - 5
        return {"messages": [RemoveMessage(id=m.id) for m in messages[:to_delete]]}
    return None


agent = create_agent(
    model=model,
    middleware=[delete_old_messages],
    checkpointer=InMemorySaver()
)

config: RunnableConfig = {"configurable": {"thread_id": "1"}}

agent.invoke({"messages": [HumanMessage("你好，我是老王")]}, config)
agent.invoke({"messages": [HumanMessage("从现在起，你叫小王")]}, config)
agent.invoke({"messages": [HumanMessage("今天天气不错")]}, config)
agent.invoke({"messages": [HumanMessage("但是深圳下了很大的雨")]}, config)
agent.invoke({"messages": [HumanMessage("北京下了很大的雪")]}, config)
final_response = agent.invoke({"messages": [HumanMessage("告诉我，你是谁？我是谁？")]}, config)

for msg in final_response["messages"]:
    msg.pretty_print()

================================== Ai Message ==================================

那今天的天气确实不太适合出门，记得带伞，路上也注意积水。
================================ Human Message =================================

北京下了很大的雪
================================== Ai Message ==================================

北京这场雪听起来不小，出行估计会受影响。路面可能结冰，开车和步行都注意安全。
================================ Human Message =================================

告诉我，你是谁？我是谁？
================================== Ai Message ==================================

我是 OpenAI 的 AI 助手。你是正在和我聊天的用户；我不知道你的姓名或其他身份信息，除非你告诉我。


### 2.4.3 RemoveMessage 实现消息删除的原理



删除对话消息的核心机制是：提交 RemoveMessage 删除指令，由 LangGraph 的 add_messages reducer 按消息 ID 执行删除。

#### 1. RemoveMessage 只是一条“删除指令”

`RemoveMessage(id=...)` 本身也是一种消息（`type` 为 `"remove"`，内容为空），但它不会被存进 state。中间件返回 `{"messages": [RemoveMessage(...), ...]}` 后，LangGraph 用 `messages` 字段的合并规则 `add_messages`（第八章 07 笔记 5.3.4）把它和现有消息合并，删除就发生在合并的时候。`add_messages` 的核心逻辑（源码 `langgraph/graph/message.py`，简化）：

```python
def add_messages(left, right):         # left：state 中现有的消息；right：本次返回的消息
    for m in left + right:             # 没有 id 的消息，先补一个 UUID
        if m.id is None:
            m.id = str(uuid.uuid4())

    # 情况 1：right 中有 RemoveMessage(id=REMOVE_ALL_MESSAGES)
    for i, m in enumerate(right):
        if isinstance(m, RemoveMessage) and m.id == REMOVE_ALL_MESSAGES:
            return right[i + 1:]       # 丢掉全部现有消息，只留它后面的消息

    # 情况 2：逐条按 id 合并
    merged = left.copy()
    merged_by_id = {m.id: i for i, m in enumerate(merged)}  # 索引：id → 在列表中的位置
    ids_to_remove = set()
    for m in right:
        if m.id in merged_by_id:                # 这个 id 已经在 state 中
            if isinstance(m, RemoveMessage):
                ids_to_remove.add(m.id)         # 删除指令：记下要删的 id
            else:
                merged[merged_by_id[m.id]] = m  # 同 id 的普通消息：替换
        else:                                   # state 中没有这个 id
            if isinstance(m, RemoveMessage):
                raise ValueError("Attempting to delete a message with an ID that doesn't exist ...")
            merged_by_id[m.id] = len(merged)
            merged.append(m)                    # 普通消息：追加
    return [m for m in merged if m.id not in ids_to_remove]
```

| 返回的内容 | 效果 | 例子 |
|---|---|---|
| `RemoveMessage(id="某条消息的 id")` | 删除这一条 | 2.4.2：删除最早的几条 |
| `RemoveMessage(id=REMOVE_ALL_MESSAGES)`，后面跟若干消息 | 清空全部消息，换成它后面的那些 | 2.4.1 的裁剪；`SummarizationMiddleware` 换成摘要 |
| `RemoveMessage(id="不存在的 id")` | 报 `ValueError` | |

`REMOVE_ALL_MESSAGES` 就是字符串 `"__remove_all__"`。直接调用 `add_messages` 看看（不请求模型）：

In [14]:
from langchain.messages import AIMessage, HumanMessage, RemoveMessage
from langgraph.graph.message import REMOVE_ALL_MESSAGES, add_messages

history = [
    HumanMessage("你好，我是老王", id="1"),
    AIMessage("你好，老王", id="2"),
    HumanMessage("今天天气不错", id="3"),
    AIMessage("是啊", id="4"),
]

# 1. 按 id 删除
result = add_messages(history, [RemoveMessage(id="1"), RemoveMessage(id="2")])
print("按 id 删除：", [m.text for m in result])

# 2. 全部清空，换成摘要 + 最近 1 条
result = add_messages(history, [RemoveMessage(id=REMOVE_ALL_MESSAGES), HumanMessage("（摘要）用户叫老王", id="s"), history[-1]])
print("全部清空：  ", [m.text for m in result])

# 3. 删除不存在的 id
try:
    add_messages(history, [RemoveMessage(id="999")])
except ValueError as e:
    print("不存在的 id：", e)

print("原列表 history 仍有", len(history), "条")

按 id 删除： ['今天天气不错', '是啊']
全部清空：   ['（摘要）用户叫老王', '是啊']
不存在的 id： Attempting to delete a message with an ID that doesn't exist ('999')
原列表 history 仍有 4 条


* 合并结果里没有 `RemoveMessage` 本身，它只是指令。原列表 `history` 也没变，`add_messages` 返回的是新列表。
* 在 Agent 中，合并后的列表成为新的 state，并被 checkpointer 存成新检查点。之后每次调用模型都从 state 读取消息，所以删除是**永久**的：这个会话之后的模型调用都看不到被删的消息。

#### 2. 消息 ID 如何与删除挂钩

删除只认 ID，不看内容。从消息进入 state 到被删除，ID 走过三步：

**① 消息进入 state 时就有了 ID**

| 消息 | ID 从哪来 |
|---|---|
| 你传入的 `HumanMessage` | 一般不指定，合并进 state 时由 `add_messages` 补一个 UUID（上面源码的第一个循环）。也可以创建时自己指定，如 `HumanMessage("...", id="my-msg-1")` |
| 模型返回的 `AIMessage` | 模型调用时生成：厂商返回了 ID 就用厂商的（如摘要小节输出中 `resp_` 开头的 ID），否则用 `lc_run--` 加上本次运行的 ID |
| 工具返回的 `ToolMessage`、摘要中间件生成的摘要消息 | 创建时没有 ID，同样由 `add_messages` 补上 |

ID 跟着消息一起存进检查点。以后每次恢复 state，同一条消息的 ID 都不变。

**② 从 state 中读出 ID，装进 RemoveMessage**

中间件拿到的 `state["messages"]` 是带 ID 的消息对象。2.4.2 中的

```python
[RemoveMessage(id=m.id) for m in messages[:to_delete]]
```

就是读出最早几条消息的 `id`，每个 ID 生成一条删除指令。`RemoveMessage` 只携带这个 ID，内容是空的。

**③ add_messages 按 ID 找到消息并删除**

中间件返回后，`add_messages` 先给 state 中现有的消息建一个“ID → 位置”的索引 `merged_by_id`，再逐条处理返回的消息：`RemoveMessage` 的 ID 在索引里查得到，就记进 `ids_to_remove`；查不到就报错。最后把 ID 在 `ids_to_remove` 中的消息过滤掉。

由此可以得出：

* 必须用 state 中那条消息的 ID。新建一条内容相同的消息去删是不行的：它的 ID 是 `None`（或者另一个值），对不上。
* 想在以后某个时刻删除某条消息，就要先“记下”它的 ID：创建消息时自己指定 ID，或者从 `get_state` 的结果里读出来，也可以存进自定义的 state 字段。
* 同一套 ID 机制也用来替换消息：返回一条 ID 相同的普通消息，会替换原来那条（第八章 07 笔记 5.3.4）。

下面在 Agent 中走一遍：先看 state 中每条消息的 ID，再用这些 ID 删除（假模型，不请求模型）：

In [17]:
from langchain_core.language_models.fake_chat_models import FakeMessagesListChatModel
from langchain.agents import create_agent
from langchain.messages import AIMessage, HumanMessage, RemoveMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph.message import add_messages

fake_model = FakeMessagesListChatModel(responses=[AIMessage("你好，老王"), AIMessage("是啊")])
id_agent = create_agent(model=fake_model, checkpointer=InMemorySaver())
id_config = {"configurable": {"thread_id": "id-demo"}}

id_agent.invoke({"messages": [HumanMessage("你好，我是老王")]}, id_config)              # 不指定 ID
id_agent.invoke({"messages": [HumanMessage("今天天气不错", id="my-msg-1")]}, id_config)  # 自己指定 ID

# ① 每条消息在 state 中都有 ID
state_messages = id_agent.get_state(id_config).values["messages"]
for m in state_messages:
    print(f"{type(m).__name__:<13} id={m.id:<48} {m.text}")

# ② 从 state 中读出第一条的 ID，连同自己指定的 ID 一起生成删除指令
# ③ add_messages 按 ID 删除（中间件返回 {"messages": [...]} 后，框架做的就是这一步）
remove = [RemoveMessage(id=state_messages[0].id), RemoveMessage(id="my-msg-1")]
print("\n删除后：", [m.text for m in add_messages(state_messages, remove)])

# 内容相同的新消息：ID 是 None，和 state 中那条对不上
print("内容相同的新消息，id =", HumanMessage("你好，我是老王").id)

HumanMessage  id=2d3a18d7-e725-4b4e-84c2-430cc2b4feda             你好，我是老王
AIMessage     id=lc_run--01a11a8c-af07-7943-8d6b-b527ae01778a-0   你好，老王
HumanMessage  id=my-msg-1                                         今天天气不错
AIMessage     id=lc_run--01a11a8c-af08-71f0-8035-a68f1088e63c-0   是啊

删除后： ['你好，老王', '是啊']
内容相同的新消息，id = None


#### 3. 为什么模型看不到被删除的消息

因为**模型收到的消息，就是调用模型那一刻 state 中的 `messages`**，而被删除的消息在这之前已经从 state 中去掉了。以在 `before_model` 中删除为例，按时间顺序看：

1. **中间件返回删除指令。** `before_model` 在 Agent 图中是一个单独的节点（名字是“中间件名.before_model”），它返回 `{"messages": [RemoveMessage(id=...)]}`。这只是交给框架的一个更新，此时 state 还没有变。
2. **这一步结束时，框架合并更新。** LangGraph 用 `add_messages` 把 `RemoveMessage` 和现有消息合并（第 1、2 小节），得到新的 `messages`：被删的消息不在里面，`RemoveMessage` 本身也不会留下。新的 state 随即存进检查点（02 笔记 2.1.6：每一步结束存一个检查点）。
3. **下一步执行 `model` 节点。** 它从 state 中读取 `messages`，构造发给模型的请求。源码（`langchain/agents/factory.py`，简化）：

```python
def model_node(state, runtime):
    request = ModelRequest(
        messages=state["messages"],                    # 直接取当前 state 中的消息
        system_message=system_message,
        ...
    )
    ...

def _execute_model_sync(request):
    messages = request.messages
    if request.system_message:
        messages = [request.system_message, *messages]  # 系统提示词加在最前面
    output = model_.invoke(messages)                    # 真正发给模型的就是这些
```

`model` 节点只从 state 中取消息，不会去别的地方找历史，所以被删的消息不会出现在请求里。

* 在 `after_model` 中删除（如 2.4.2）是同样的道理，只是删除发生在模型调用之后：本次调用已经看过这些消息，从下一次调用模型开始就看不到了。
* 之后再调用 `invoke` 时，Agent 从检查点恢复的也是删除后的 state，所以删除对这个会话是永久的。2.4.1 的裁剪也是这样：它在 `before_model` 中返回 `RemoveMessage(id=REMOVE_ALL_MESSAGES)`，被裁掉的消息不只是这次没发给模型，而是从 state 中删掉了，所以最后只打印出 6 条。

下面用 `stream_mode="updates"`（第七章 08 笔记）逐个打印节点返回的更新，再用一个只打印、不修改的 `wrap_model_call` 中间件，显示 `model` 节点实际发给模型的消息（假模型，不请求模型）：

In [18]:
from langchain_core.language_models.fake_chat_models import FakeMessagesListChatModel
from langchain.agents import AgentState, create_agent
from langchain.agents.middleware import before_model, wrap_model_call
from langchain.messages import AIMessage, HumanMessage, RemoveMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.runtime import Runtime


@before_model
def delete_first(state: AgentState, runtime: Runtime) -> dict | None:
    """消息达到 3 条时，删除最早的一条"""
    if len(state["messages"]) >= 3:
        return {"messages": [RemoveMessage(id=state["messages"][0].id)]}
    return None


@wrap_model_call
def show_request(request, handler):
    """只打印 model 节点发给模型的消息，不做修改"""
    print(f"    model 节点发给模型：{[m.text for m in request.messages]}")
    return handler(request)


fake_model = FakeMessagesListChatModel(responses=[AIMessage("你好，老王"), AIMessage("是啊")])
why_agent = create_agent(model=fake_model, middleware=[delete_first, show_request], checkpointer=InMemorySaver())
why_config = {"configurable": {"thread_id": "why-demo"}}
print("Agent 图中的节点：", list(why_agent.nodes))

print("\n第 1 次调用：")
why_agent.invoke({"messages": [HumanMessage("你好，我是老王")]}, why_config)

print("\n第 2 次调用，逐个打印节点返回的更新：")
for chunk in why_agent.stream({"messages": [HumanMessage("今天天气不错")]}, why_config, stream_mode="updates"):
    for node, update in chunk.items():
        print(f"  {node} 返回：{[type(m).__name__ for m in update['messages']]}")

print("\n最终 state：", [m.text for m in why_agent.get_state(why_config).values["messages"]])

Agent 图中的节点： ['__start__', 'model', 'delete_first.before_model']

第 1 次调用：
    model 节点发给模型：['你好，我是老王']

第 2 次调用，逐个打印节点返回的更新：
  delete_first.before_model 返回：['RemoveMessage']
    model 节点发给模型：['你好，老王', '今天天气不错']
  model 返回：['AIMessage']

最终 state： ['你好，老王', '今天天气不错', '是啊']


* 第 2 次调用时，state 中有 3 条消息（第 1 次的问答 + 新问题），达到删除条件。`delete_first.before_model` 节点只返回了一条 `RemoveMessage`。
* 这一步结束后，框架按 ID 把“你好，我是老王”从 state 中去掉。接着 `model` 节点从 state 读取消息，发给模型的就只剩 2 条。
* 最终 state 里既没有被删的消息，也没有 `RemoveMessage`。

> 模型看不到，不代表数据已经不存在：之前存下的检查点是当时的快照，仍然包含被删的消息，用 `get_state_history` 能看到（02 笔记 2.1.6）。如果出于隐私原因必须彻底删除，要用 `checkpointer.delete_thread(thread_id)` 删除整个会话。

### 2.4.4 消息摘要

把早期历史压缩成摘要，再替换原始消息。

消息裁剪和删除都会导致上下文缺失，影响回答质量和用户体验。和它们相比，摘要是更适合长会话的

折中方案：保语义，不保原文。官方推荐内置 SummarizationMiddleware 。上一章已有讲解。

In [10]:
from langchain.agents.middleware import SummarizationMiddleware
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver


# 创建带摘要中间件的 Agent
agent = create_agent(
    model=model,
    tools=[],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=[
                ("tokens", 100),  # 超过 100 tokens 就摘要
            ],
            keep=("messages", 2),
            summary_prompt="对历史消息摘要，消息列表如下\n{messages}",

        )
    ]
)

config = {"configurable": {"thread_id": "1"}}

print("\n进行多轮对话...")
conversations = [
    "我叫张三，是工程师。这里是一段非常长非常长的废话..." * 20, # 强制撑爆 100 tokens
    "请总结一下我的信息"
]

for msg in conversations:
    response = agent.invoke(
        {"messages": [{"role": "user", "content": msg}]},
        config=config
    )
    for msg in response["messages"]:
        msg.pretty_print()
    print("*" * 50)


进行多轮对话...
================================ Human Message =================================

我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...
================================== Ai Message ==================================

[{'id': 'rs_0c295b6493fa2cdd016ac739dd1d08819ab8c02324eac39c78', 'summary': [], 'type': 'reasoning', 'content': [], 'encrypted_content': 'gAAAAABqxznd7EE6zPo6_ehEfWPQs2BksWqfufKGcGjacFbE6cCWgoRSq4s3QT-ON1R0wZGdAD4gmjUwPK203Jjt-Ai5ilHZw9UtJdaWnRRjHBjDCisVkZq_36mf6lWmVi7b3GFTMOT0A9OgHO

In [11]:
from rich import print as rprint

final_state = agent.get_state(config)

rprint(final_state)

StateSnapshot(
    values={
        'messages': [
            HumanMessage(
                content='Here is a summary of the conversation to 
date:\n\n历史消息摘要：\n\n用户张三，自称工程师。消息主要内容是反复重复“我叫张三，是工程师”以及“这里是一段非常长非
常长的废话”，没有提供其他具体需求或有效信息。',
                additional_kwargs={'lc_source': 'summarization'},
                response_metadata={},
                id='8a8eb9a0-a504-4e1c-b9dd-ad84e949d78a'
            ),
            AIMessage(
                content=[
                    {
                        'id': 'rs_0c295b6493fa2cdd016ac739dd1d08819ab8c02324eac39c78',
                        'summary': [],
                        'type': 'reasoning',
                        'content': [],
                        'encrypted_content': 
'gAAAAABqxznd7EE6zPo6_ehEfWPQs2BksWqfufKGcGjacFbE6cCWgoRSq4s3QT-ON1R0wZGdAD4gmjUwPK203Jjt-Ai5ilHZw9UtJdaWnRRjHBjDCi
sVkZq_36mf6lWmVi7b3GFTMOT0A9OgHOiOSmnOh9p8Wyqp89nkbWtP25EpAAy8JpiXJWqBuCzuf3Dx8rEiti-mambwZGRJAaDgft2L9KlomnPDqeoCb
ENqxCN_kbPO-oswogGXrntvlSVbvZKu9Xd0CS52xlCeL3r0iL693WnzGeDcVc0mSVpnT6TKfn00rXoYYOcHB8n7SRWz0sWdwYTYtJKoqpLzpZ_0ycDC
FE4pdYo_KPEDX6Y0WEwnJvnvh54KOIS-W9a8FPPFj4UZFcjoAt39DzK9COzBABW7qV2CKsJAD9HCQkl_3IJaHJgNP6YFl2AIGU4rbDcmdPtSjiVB7g8
kq-sw-pj_WjDnxzY8AplNz8tDR8yHUNfbxp-cmH0oQFd9C2Xfr9kSklc1FpirMCdR61WBuKG-AZW3-FoQpcJmf0NHaxnp78KTIHdj-PWwzSBfbWnFXx
Vgja-mOZjoL0fCy6n1N-LltEjOXS0quv9USLIJe0F9jbuXW699pIypHFqCWYwQgAgmFokPtcL0S6Tmfn8jBKYn7jvJa76Bf33n_jGX5MEVqpJnc7o8P
4PGSxzAbJY5Kmn5vaSebCaQ_Sq2v8n2i0CcdWtyLCiQEYuzOeXNoV6K7XX0l6gVUWIJiSkGSJVlGw3P9MmTj4IW2Pl81FS1LPEzdj2LEmyvC3t2L6aq
d89ByVy0R0Qt5s-1fezkCrdgP5-Cm5FYKpSqYMGsZ1dtSfp8E2wthrrHWphlamwiq3aS1aoXxoLFBtDFEy0Xyq3JWrFhqCvE2-CxHWuKlJQXh3Lew52
GadG6LpRbZA7NboQ14Dtyc7mkWfr8v04kr67gsofsIuxJn0I3fnq1D1zNDpQZdE7nwT7TpG5ikw8kMhhkMud9ED6YZd6wbCnU-ImjMTjZpn3vYhz3_G
hfoJymn3z0vius2g7MUWtYSWBlWimGO80TfGi1xCYMraqiiO-lFYASVW3v_OrMmgwayNezpBnxMHv6r7Y5ofQxURL5S3Zchy8i4SPdKrRKltHuPsF0G
ISwaK-pJMfIOvZ61oNcYV5mO0Jrq-jTgirRAI3LbAEncydzUlYf-EaaXulfsWzyfdTRmmSsRVHGahN-aKcH9g5FlMqmrcGEFmUfu9b7PuY20UGFEi5j
IbP0g3k3rxPJFtGYoAJ2R0Ts4GQe3lclu57kGwBmu1NqxO4CBokJBj9bJjOqiK3ZHo3PKMzPI7AzPOj7xVP3bsZ-keHQKz4eUUFDEqv8L_37ab0eRRM
lBWwn6ytiRMYHFECSfa5E47RgLt6pSvWUU_s2Nxj8J24_Ghl1s8ihlxaY5I7ISGzkzChi8s-MwhTAX633fDYNA4RFgclvlZmVDeMf3Jy16vm2jPfKPy
oxUF1bhmX6kw=='
                    },
                    {
                        'type': 'text',
                        'text': '请说明你希望我如何处理这段文字，例如：\n\n- 总结或提炼重点\n- 删除重复内容\n- 
统计字数、句子或重复次数\n- 改写成更正式的自我介绍\n- 翻译成其他语言\n- 检测是否存在重复文本',
                        'annotations': [],
                        'id': 'msg_0c295b6493fa2cdd016ac739dd6e3c819a9889f1ceecd32c3a',
                        'phase': 'final_answer'
                    }
                ],
                additional_kwargs={},
                response_metadata={
                    'id': 'resp_0c295b6493fa2cdd016ac739d45474819ab8693199bfbd50fd',
                    'created_at': 1791441364.0,
                    'metadata': {},
                    'model': 'gpt-5.6-sol',
                    'object': 'response',
                    'service_tier': 'default',
                    'status': 'completed',
                    'model_provider': 'openai',
                    'model_name': 'gpt-5.6-sol'
                },
                id='resp_0c295b6493fa2cdd016ac739d45474819ab8693199bfbd50fd',
                tool_calls=[],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 4766,
                    'output_tokens': 106,
                    'total_tokens': 4872,
                    'input_token_details': {'cache_creation': 0, 'cache_read': 3840},
                    'output_token_details': {'reasoning': 35}
                }
            ),
            HumanMessage(
                content='请总结一下我的信息',
                additional_kwargs={},
                response_metadata={},
                id='c04bd75c-b814-4c2d-8e52-1ea7ef33edf7'
            ),
            AIMessage(
                content=[
              

#### 补充：为什么第一次打印 response["messages"] 时，长消息还在

![SummarizationMiddleware 在什么时候摘要](assets/07-summarization-timeline.svg)

原因有两点：

1. **摘要发生在调用模型之前。** `SummarizationMiddleware` 挂在 `before_model` 上（第八章 02 笔记）：每次调用模型前，检查一次当前的消息，需要时把旧消息换成摘要。`invoke` 返回的是这次运行结束时的 state，所以第一次打印的是“第 1 次调用模型前没有摘要”的结果。
2. **最近 `keep` 条消息总是保留原文。** 第 1 次调用前，state 中只有 1 条消息，就是这条长消息。它确实超过了 100 tokens，满足 `trigger`；但 `keep=("messages", 2)` 要求保留最近 2 条原文，总共只有 1 条，没有“更早的消息”可以摘要，所以中间件什么都不做。这样设计也合理：这条消息是用户当前的问题，要是把它换成摘要，模型就看不到原问题了。

第 2 次调用前，state 中有 3 条消息：保留最近 2 条（AI 回复 1、“请总结一下我的信息”），最早的长消息被摘要。所以第二次打印中，长消息变成了以 `Here is a summary of the conversation to date:` 开头的 `HumanMessage`。

`before_model` 的源码（`langchain/agents/middleware/summarization.py`，简化）：

```python
def before_model(self, state, runtime):
    messages = state["messages"]
    total_tokens = self.token_counter(messages)
    if not self._should_summarize(messages, total_tokens):   # ① 是否满足 trigger
        return None

    cutoff_index = self._determine_cutoff_index(messages)   # ② 去掉最近 keep 条后，前面还剩几条
    if cutoff_index <= 0:
        return None                                         # 没有可摘要的旧消息：不做修改

    messages_to_summarize = messages[:cutoff_index]          # 要摘要的旧消息
    preserved_messages = messages[cutoff_index:]             # 保留原文的最近消息
    summary = self._create_summary(messages_to_summarize)   # 调用摘要模型
    return {
        "messages": [
            RemoveMessage(id=REMOVE_ALL_MESSAGES),          # 清空现有消息（见 2.4.3）
            HumanMessage(f"Here is a summary of the conversation to date:\n\n{summary}"),
            *preserved_messages,
        ]
    }
```

用上面的参数创建一个 `SummarizationMiddleware`，对两次调用前的消息分别做 ①② 两步检查。下划线开头的是内部方法，这里只用来观察，项目中不要调用（不请求模型）：

In [16]:
from langchain.agents.middleware import SummarizationMiddleware
from langchain.messages import AIMessage, HumanMessage

checker = SummarizationMiddleware(model=model, trigger=[("tokens", 100)], keep=("messages", 2))
long_text = "我叫张三，是工程师。这里是一段非常长非常长的废话..." * 20

before_round1 = [HumanMessage(long_text)]
before_round2 = [
    HumanMessage(long_text),
    AIMessage("请说明你希望我如何处理这段文字……"),
    HumanMessage("请总结一下我的信息"),
]

for name, msgs in [("第 1 次调用前", before_round1), ("第 2 次调用前", before_round2)]:
    tokens = checker.token_counter(msgs)
    triggered = checker._should_summarize(msgs, tokens)
    cutoff = checker._determine_cutoff_index(msgs)
    print(f"{name}：{len(msgs)} 条消息，约 {tokens} tokens，满足 trigger：{triggered}，"
          f"要摘要 {cutoff} 条，保留原文 {len(msgs) - cutoff} 条")

第 1 次调用前：1 条消息，约 139 tokens，满足 trigger：True，要摘要 0 条，保留原文 1 条
第 2 次调用前：3 条消息，约 156 tokens，满足 trigger：True，要摘要 1 条，保留原文 2 条


* 这里的 token 数是 `SummarizationMiddleware` 默认的估算值（按字符数估算），不是模型实际计费的 token 数。
* `keep` 也可以按 token 数设置，如 `keep=("tokens", 50)`。切分时还会避免把 `AIMessage` 和它对应的 `ToolMessage` 拆开，保证工具调用和结果在同一边。
* 输出中 AI 消息显示成一长串列表，是因为 `gpt-5.6-sol` 返回的内容包含推理（`reasoning`）块和文字块。只想看文字时，可以打印 `msg.text`。

### 2.4.5 自定义过滤策略
通过 中间件 可以随意更改消息列表，因此可以实现任意的过滤策略。
此处不再演示。